In [ ]:
# ## Imports

import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    recall_score, precision_score, accuracy_score,
)
from xgboost import XGBClassifier

In [ ]:
# ## 1 · Data Splitting

def split_dataset(csv_path, save_prefix=None, test_size=0.2, val_size=0.2, random_state=42):
    """
    Loads a CSV, creates stratified train/val/test splits by cardio × gender.

    Args:
        csv_path     : path to CSV file
        save_prefix  : if provided, saves to ../data/test_train_val_sets/{prefix}_*.csv
        test_size    : proportion for test set (default 0.2)
        val_size     : proportion of train for val (default 0.2 → ~16% of total)
        random_state : random seed

    Returns:
        train_df, val_df, test_df
    """
    df = pd.read_csv(csv_path)
    df["stratify"] = df["cardio"].astype(str) + "_" + df["gender"].astype(str)

    train_df, test_df = train_test_split(
        df, test_size=test_size, stratify=df["stratify"], random_state=random_state,
    )
    train_df, val_df = train_test_split(
        train_df, test_size=val_size, stratify=train_df["stratify"], random_state=random_state,
    )

    print(f"Dataset    : {csv_path}")
    print(f"Train      : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
    print(f"Val        : {len(val_df)}   ({len(val_df)/len(df)*100:.1f}%)")
    print(f"Test       : {len(test_df)}  ({len(test_df)/len(df)*100:.1f}%)")

    if save_prefix:
        base = "../data/test_train_val_sets"
        train_df.to_csv(f"{base}/{save_prefix}_train.csv", index=False)
        val_df.to_csv(f"{base}/{save_prefix}_val.csv",     index=False)
        test_df.to_csv(f"{base}/{save_prefix}_test.csv",   index=False)
        print(f"Saved to {base}/{save_prefix}_*.csv")

    return train_df, val_df, test_df


train_df, val_df, test_df = split_dataset(
    "../data/processed/cardio_baseline_clean.csv",
    save_prefix="cardio_baseline",
)

In [ ]:
# ## 2 · Prepare X / y

X_train = train_df.drop(columns=["cardio", "stratify"])
y_train = train_df["cardio"]

X_val = val_df.drop(columns=["cardio", "stratify"])
y_val = val_df["cardio"]

X_test = test_df.drop(columns=["cardio", "stratify"])
y_test = test_df["cardio"]

female_mask_test = test_df["gender"].values == 0
male_mask_test   = test_df["gender"].values == 1

In [ ]:
# ## 3 · Train Baseline Model

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    early_stopping_rounds=10,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)

with open("baseline_models/cardio_xgb_baseline_model.pkl", "wb") as f:
    pickle.dump(model, f)